# Introduction to Software Engineering
(Or How I Learned To Stop Worrying and Love Doc Strings)
----

----
By Adam A Miller (Northwestern/CIERA/SkAI)  
16 Sept 2026

**Version 0.1**

We continue with today's theme #protectingmyselffrommyself this afternoon. 

This lecture covers the things you can do to help someone else (including you six months from now!) run your code. 

## Learning Objectives
1. **Docstrings** — so the next person doesn't have to guess
2. **Unit tests** — so a change to one part doesn't silently break another
3. **Packages** — so the whole thing can be installed and reused

## Problem 1) Docstrings

Docstrings are strings placed immediately after a function, class, or module definition, describing what it does, what it expects, and what it returns.

They're written between triple quotes, right after the definition line. 

Comments are not docstrings. 

Any docstring can be pulled up directly with a `?` in `IPython` (no need to open the source file).

Do you remember how to generate random numbers from a multivariate Gaussian distribution?  

(me either, but no sweat)

In [1]:
import numpy as np

np.random.Generator.multivariate_normal?

Signature:     
np.random.Generator.multivariate_normal(
    self,
    mean,
    cov,
    size=None,
    check_valid='warn',
    tol=1e-08,
    *,
    method='svd',
)
Call signature: np.random.Generator.multivariate_normal(*args, **kwargs)
Type:           cython_function_or_method
String form:    <cyfunction Generator.multivariate_normal at 0x10df9cc40>
Docstring:     
multivariate_normal(mean, cov, size=None, check_valid='warn',
                    tol=1e-8, *, method='svd')

Draw random samples from a multivariate normal distribution.

The multivariate normal, multinormal or Gaussian distribution is a
generalization of the one-dimensional normal distribution to higher
dimensions.  Such a distribution is specified by its mean and
covariance matrix.  These parameters are analogous to the mean
(average or "center") and variance (the squared standard deviation,
or "width") of the one-dimensional normal distribution.

Parameters
----------
mean : 1-D array_like, of length N
    Mean of th

The page of documentation that pops up here — parameters, return values, examples — is not written separately somewhere. It's rendered directly from the function's docstring.

As a quick aside -- 

Tools like Sphinx and `pydoc` scan a package's docstrings and generate an HTML documentation site from them automatically. 

(this is really really cool - if you write docstrings then you've also written nice documentation!)

A page like numpy's own online documentation is, structurally, the same docstring you just looked at with `?` — just rendered as a web page instead of printed in a terminal.

I'm going to give you 3 sec to figure out what the next function does. 

Ready, set, go...

In [ ]:
def convert(t, u):
    if u == "F":
        return (t - 32) * 5 / 9
    return t * 9 / 5 + 32

What does `u` stand for? What does this return if `u` isn't `"F"`? Is the output Celsius or Fahrenheit in that case? Nothing here answers that — you'd have to read the body and reason it out.

Again, you get 3 seconds, but now there are docstrings:

In [ ]:
def convert(temperature, unit):
    """
    Convert a temperature between Celsius and Fahrenheit.

    Parameters
    ----------
    temperature : float
        The temperature to convert.
    unit : str
        The unit of `temperature`, either "C" or "F". The result is
        returned in the other unit.

    Returns
    -------
    float
        The converted temperature.
    """
    if unit == "F":
        return (temperature - 32) * 5 / 9
    return temperature * 9 / 5 + 32

Same code. The only difference is whether a reader has to run the function in their head to find out what it does, or can just read the first four lines.

There is no "standard" for docstrings, but I like the `NumPy` convention, and typically follow that: a `Parameters` section and a `Returns` section, each with types.

Google-style and reST-style docstrings are also common and use a slightly different layout. None is enforced by Python itself; picking one and using it consistently across a project matters more than which one you pick.

One more aside – type hints are a related but separate tool: they state a parameter or return type in the function signature itself, and can be checked by tools like `mypy` before the code ever runs.

In [ ]:
def convert(temperature: float, unit: str) -> float:
    """
    Convert a temperature between Celsius and Fahrenheit.

    Parameters
    ----------
    temperature : float
        The temperature to convert.
    unit : str
        The unit of `temperature`, either "C" or "F". The result is
        returned in the other unit.

    Returns
    -------
    float
        The converted temperature.
    """
    if unit == "F":
        return (temperature - 32) * 5 / 9
    return temperature * 9 / 5 + 32

Hints and docstrings complement each other, but they are not the same thing.

## Problem 2) Unit tests

A unit test is a small, automated check that a specific piece of code — usually one function or method — behaves as expected.

The point isn't to prove the code is perfect. It's to catch the moment a *later* change breaks something that used to work.

What would a test look like for `convert()`?

In [2]:
def test_convert_freezing_point():
    assert convert(0, "C") == 32

We know that $0^\circ$ Celcius is equal to 32. If at any point the code produces an output that disagrees, we can instantly catch that and begin debugging. 

(I'll admit, it's hard for me to get excited about unit tests. But they can be a lifesaver, especially if you port your code to a super computer)

There are two common ways to set up unit tests in python: `unittest` & `pytest`.

In [3]:
# unittest
import unittest

class TestConvert(unittest.TestCase):
    def test_freezing_point(self):
        self.assertEqual(convert(0, "C"), 32)

In [4]:
# pytest — a plain function and a bare assert; no class required
def test_freezing_point():
    assert convert(0, "C") == 32

`pytest` has become the more common choice in scientific Python, because it is a little easier to write.

Multiple tests are better than no tests

In [5]:
def test_convert_catches_unit_confusion():
    # if convert() silently swapped Celsius and Fahrenheit,
    # this would fail loudly instead of producing a wrong
    # number three functions downstream
    assert convert(100, "C") == 212
    assert convert(212, "F") == 100

A test like this exists specifically to catch the failure mode where a unit gets flipped, or a formula gets transcribed backwards — the kind of bug that doesn't raise an error, it just produces a plausible-looking wrong number.

Tests can also be used to check failure modes

In [6]:
import pytest

def test_convert_rejects_unknown_unit():
    with pytest.raises(ValueError):
        convert(100, "K")

This only passes once `convert` is actually changed to raise a `ValueError` for units it doesn't recognize, rather than silently falling through to the Fahrenheit branch as the current version does. A test can specify a behavior that doesn't exist yet.

A test suite that only checks "did the function run" or "is the attribute equal to what I passed in" will pass even when the underlying logic is wrong.

The useful question when writing a test isn't "what does this code currently do" — it's "what's a case where this code *could plausibly be wrong*, and would I actually notice?"

## Problem 3) Packages

Once code is organized into classes, documented, and tested, you can go for the real gold star and make your software installable. 

Then anyone, anywhere can import the software (and not be stuck on a single machine in a single directory). 

A package will let others install your code with `pip`, and gives it a natural home for its tests and documentation.

Suppose we wanted to create a package for our N body code from a few days ago. A useful structure would look something like this: 

`digital-orrery/`  
$~~~~~~~~$├── `digital_orrery/`  
$~~~~~~~~$│$~~~~~~~~$$~~~~~~~~$├── `__init__.py`  
$~~~~~~~~$│$~~~~~~~~$$~~~~~~~~$├── `body.py`  
$~~~~~~~~$│$~~~~~~~~$$~~~~~~~~$└── `universe.py`  
$~~~~~~~~$├── `tests/`  
$~~~~~~~~$│$~~~~~~~~$$~~~~~~~~$├── `__init__.py`  
$~~~~~~~~$│$~~~~~~~~$$~~~~~~~~$└── `test_body.py`  
$~~~~~~~~$├── `pyproject.toml`  
$~~~~~~~~$├── `README.md`  
$~~~~~~~~$└── `LICENSE`  

Some of this is obvious based on previous lectures, but what goes in `__init__.py`?

On its own this file can be empty — its presence is what tells Python the directory is a package. In practice, it's usually used to control what's available at the top level of the package.

    # digital_orrery/__init__.py  
    from .body import Body
    from .universe import Universe

This lets a user write `from digital_orrery import Body` instead of `from digital_orrery.body import Body` — a small convenience, but one that shapes what your package's public interface looks like.

And what about `pyproject.toml`?

    [project]
    name = "digital_orrery"
    version = "0.1.0"
    dependencies = ["numpy"]
    
    [build-system]
    requires = ["setuptools"]
    build-backend = "setuptools.build_meta"

This has largely replaced `setup.py` as the standard place to declare a package's name, version, and dependencies. If you see `setup.py`-only instructions in an older tutorial, `pyproject.toml` is the current equivalent.

How do I install this package?

In [7]:
pip install -e .

Obtaining file:///Users/adamamiller/astronomy/OnboardOrientation/2026-onboard/Day5
ERROR: file:///Users/adamamiller/astronomy/OnboardOrientation/2026-onboard/Day5 does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
Note: you may need to restart the kernel to use updated packages.


The `-e` installs it in *editable* mode: Python treats it as an installed package, importable from anywhere, but changes to the source files take effect immediately, without reinstalling.

This is the step that makes `from digital_orrery import Universe` work in a notebook that isn't sitting in the same folder as the code.

At this stage we can pull together everything that we discuss today:

- `tests/` is where the unit tests from Problem 2 go — `pytest` looks for files named `test_*.py` there automatically

- Every function and class in `body.py` and `universe.py` should carry the kind of docstring from Problem 1 — this is what a `pip`-installed package's documentation is actually built from

- `environment.yml`, from this morning, usually sits alongside `pyproject.toml` at the top level

## A package is a checkpoint, not an end point

`README.md` explains what the package does and how to install it. `LICENSE` states the terms under which anyone else can use it. Neither is optional if the goal is for someone outside your group to actually use this.

With those in place, along with a repository on GitHub, this is a tool that can be shared, cited, and built on by someone who has never spoken to you.